# FLUXION — F06 Media Generator (v1)
> 29 immagini realistiche per 4 verticali PMI: Fitness / Fisioterapia / Odontoiatrica / Veicoli
> Architettura v19: T5+CLIP pre-encode → FLUX float16 → sequential_cpu_offload (zero bitsandbytes)
> Research: CoVe 2026 — standard fotografici world-class (TrueCoach, Jane App, AACD, Mitchell1)
> Output: /kaggle/working/f06-media/{verticale}/

In [ ]:
# CELLA 1 — INSTALL (versioni pinned — stessa v19 validata)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'diffusers>=0.30.3',
     'transformers==4.46.3',
     'accelerate>=0.34.0',
     'sentencepiece', 'protobuf', 'safetensors',
    ],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('pip install fallito')
print('Dipendenze installate OK — NO bitsandbytes (v19 pattern)')

In [ ]:
# CELLA 2 — VERIFICA GPU + PATH MODELLO
import torch, os
from pathlib import Path
import pkg_resources

for pkg in ['diffusers', 'transformers', 'accelerate']:
    print(f'  {pkg}: {pkg_resources.get_distribution(pkg).version}')
print(f'torch: {torch.__version__}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU non disponibile')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

MODEL_PATH = '/kaggle/input/models/sanderland/flux.1-schnell/pytorch/hf/1'
p = Path(MODEL_PATH)
if not p.exists():
    for root, dirs, files in os.walk('/kaggle/input'):
        if any('schnell' in f.lower() or 'flux' in f.lower() for f in files + dirs):
            print(f'Trovato in: {root}')
            MODEL_PATH = root
            break
    else:
        raise RuntimeError(f'Modello non trovato.')
print(f'Modello: {MODEL_PATH}')

In [ ]:
# CELLA 3 — PROMPTS F06 (da CoVe 2026 deep research)
# Fonte: .claude/cache/agents/f06-media-generation-research.md
# JSON: _bmad-output/f06-media-prompts.json
# Formato: {filename: (prompt, negative_prompt, width, height, seed, verticale)}

PROMPTS = {
    # ── FITNESS / PALESTRA ──────────────────────────────────────────────────
    'fitness/fitness_001_progress_before.png': (
        'Professional fitness progress photo, athletic male client, 30s, front view, '
        'standing upright relaxed pose, neutral white wall background, even soft studio lighting, '
        'grey athletic shorts, bare chest, full body frame head to feet, '
        'clinical documentation style, realistic photography, sharp focus',
        'cartoon, anime, blurry, watermark, text, flexing pose, gym equipment, colored background, dramatic shadows',
        832, 1216, 1001),

    'fitness/fitness_002_progress_after.png': (
        'Professional fitness transformation after photo, athletic male client, 30s, front view, '
        'standing upright, visible muscle definition improvement, neutral white wall background, '
        'consistent even lighting, grey athletic shorts, bare chest, full body frame, '
        'before-after comparison documentation style, realistic photography',
        'cartoon, anime, dramatic lighting, excessive muscle, unrealistic proportions, colored gels, text, watermark',
        832, 1216, 1001),

    'fitness/fitness_003_progress_lateral.png': (
        'Professional fitness progress photo, athletic female client, 35s, side lateral view, '
        'right profile, standing straight relaxed, neutral grey background, '
        'soft diffused window lighting, black sports bra, fitted shorts, full body frame, '
        'clinical body composition documentation, realistic photography',
        'cartoon, anime, overly sexualized, dramatic shadows, artistic pose, colored backdrop, blurry, text, watermark',
        832, 1216, 1002),

    'fitness/fitness_004_before_after_grid.png': (
        'Side-by-side fitness transformation comparison, same male client front view, '
        'left panel BEFORE showing higher body fat, right panel AFTER showing lean muscle definition, '
        'consistent white background both panels, identical pose and framing, '
        'professional documentation photography, clean layout, realistic',
        'cartoon, extreme bodybuilding, unrealistic proportions, different lighting between panels, text, watermark',
        1216, 832, 1001),

    'fitness/fitness_005_workout_technique.png': (
        'Personal trainer observing client performing barbell squat with correct form, '
        'modern gym interior, professional overhead LED lighting, trainer beside client, '
        'focus on form documentation, mid-action realistic photography, high resolution, '
        'professional fitness coaching environment',
        'cartoon, anime, poor form, dangerous position, blurry motion, stock photo style, watermark, text',
        1216, 832, 1003),

    'fitness/fitness_006_body_measurement.png': (
        'Professional body measurement documentation photo, female client, '
        'fitness trainer measuring waist circumference with yellow measuring tape, '
        'neutral background, clinical bright lighting, both subjects professional attire, '
        'realistic medical fitness documentation style, medium shot',
        'cartoon, anime, inappropriate framing, blurry, dramatic lighting, text, watermark',
        1216, 832, 1004),

    'fitness/fitness_007_monthly_timeline.png': (
        'Fitness progress timeline grid, 3-month transformation series, same athletic male subject, '
        'front view in all three photos, January February March labels, '
        'consistent white background and lighting across all shots, '
        'visible gradual body composition improvement, professional documentation layout, realistic photography',
        'cartoon, dramatic before-after, unrealistic rapid transformation, different backgrounds, watermark',
        1216, 832, 1001),

    # ── FISIOTERAPIA / RIABILITAZIONE ───────────────────────────────────────
    'fisioterapia/physio_001_postural_anterior.png': (
        'Clinical physiotherapy postural assessment photo, male patient 40s standing upright, '
        'anterior frontal view, neutral dark grey background, bright uniform clinical lighting, '
        'patient wearing black shorts only, white anatomical markers on bilateral shoulders and hips, '
        'physiotherapy clinic setting, medical documentation photography style, full body frame',
        'cartoon, anime, dramatic shadows, colorful background, casual clothing, gym setting, text, watermark',
        832, 1216, 2001),

    'fisioterapia/physio_002_postural_lateral.png': (
        'Clinical physiotherapy lateral posture assessment, female patient 35s, right side profile view, '
        'standing straight, dark neutral background, clinical overhead lighting, '
        'anatomical markers on ear, shoulder, hip, knee and ankle, '
        'vertical plumb line reference visible on the side, physiotherapy clinic, full body',
        'cartoon, artistic lighting, colorful background, casual setting, blurry, text, watermark',
        832, 1216, 2002),

    'fisioterapia/physio_003_rom_assessment.png': (
        'Physiotherapy range of motion documentation photo, male patient performing maximum shoulder flexion, '
        'physical therapist beside patient with goniometer measuring joint angle, '
        'clinical room white walls, bright even lighting, patient in grey shorts and t-shirt, '
        'close-medium shot showing arm elevation angle clearly, clinical measurement documentation, realistic',
        'cartoon, anime, blurry, casual setting, dark lighting, artistic style, text, watermark',
        1216, 832, 2003),

    'fisioterapia/physio_004_treatment_session.png': (
        'Professional physiotherapy treatment session, female therapist performing manual therapy '
        'on male patient lying on treatment table, modern physiotherapy clinic, '
        'clean white and blue color scheme, professional medical lighting, '
        'both wearing professional attire, clinical environment, realistic photography, medium wide shot',
        'cartoon, anime, inappropriate contact, unprofessional setting, dim lighting, dramatic shadows, text, watermark',
        1216, 832, 2004),

    'fisioterapia/physio_005_before_after_posture.png': (
        'Physiotherapy posture improvement comparison, same male patient side lateral view, '
        'left panel BEFORE showing forward head posture and rounded shoulders, '
        'right panel AFTER showing corrected upright posture, '
        'identical dark background both panels, same anatomical markers visible, '
        'clinical documentation layout, realistic medical photography',
        'cartoon, exaggerated before, unrealistic correction, different backgrounds, artistic style, watermark',
        1216, 832, 2001),

    'fisioterapia/physio_006_exercise_rehab.png': (
        'Patient performing therapeutic exercise in physiotherapy clinic, female patient 50s '
        'doing resistance band shoulder external rotation exercise, '
        'physical therapist supervising with correct form, modern rehabilitation gym, '
        'professional clinical lighting, clean modern clinic interior, realistic photography',
        'cartoon, anime, incorrect form, gym commercial style, dramatic lighting, text, watermark',
        1216, 832, 2005),

    'fisioterapia/physio_007_swelling_doc.png': (
        'Clinical documentation photo of ankle swelling assessment, close-up of patient ankle with mild edema, '
        'yellow measuring tape around ankle circumference for girth measurement, '
        'clinical white background, bright even medical lighting, centimeter scale visible, '
        'professional clinical photography, medical documentation style',
        'cartoon, gruesome, blurry, dark, artistic, text, watermark, colored background',
        1024, 1024, 2006),

    # ── ODONTOIATRICA / STUDIO DENTISTICO ───────────────────────────────────
    'odontoiatrica/dental_001_smile_before.png': (
        'Professional dental before photo, patient frontal smile, retracted lips showing full dentition, '
        'grey studio background, ring flash dental photography lighting, '
        'slight crowding and discoloration visible, clinical documentation style, '
        'sharp focus on teeth, realistic dental photography, AACD standard',
        'cartoon, anime, perfect white teeth in before, blurry, dark background, stock photography smile, watermark, text',
        1024, 1024, 3001),

    'odontoiatrica/dental_002_smile_after.png': (
        'Professional dental after photo, patient frontal retracted smile showing full dentition post-cosmetic treatment, '
        'grey studio background, ring flash dental photography lighting, '
        'beautifully aligned and whitened teeth, veneer work visible, '
        'clinical documentation AACD standard, sharp dental detail, realistic dental photography',
        'cartoon, anime, unrealistic hyper-white teeth, blurry, different lighting from before, artistic filter, watermark, text',
        1024, 1024, 3001),

    'odontoiatrica/dental_003_before_after_portrait.png': (
        'Side-by-side dental smile transformation, patient portrait, '
        'left panel BEFORE showing stained and slightly crooked smile, '
        'right panel AFTER showing beautiful cosmetic dentistry result, '
        'both panels identical neutral grey background and professional soft lighting, '
        'consistent head position, realistic dental photography, cosmetic case presentation',
        'cartoon, different lighting between panels, unrealistic whitening, text cluttered, stock photo, watermark',
        1216, 832, 3001),

    'odontoiatrica/dental_004_intraoral_retracted.png': (
        'Clinical intraoral dental photo, retracted frontal view of patient dentition, '
        'both arches in occlusion, lip retractors visible holding lips open, '
        'black contrasting background, ring flash lighting eliminating shadows, '
        'slight wear patterns visible on enamel, sharp macro detail of all teeth, '
        '1:2 clinical magnification, professional dental documentation photography',
        'cartoon, saliva visible, blurry, dark exposure, artistic framing, colored background, text, watermark',
        1216, 832, 3002),

    'odontoiatrica/dental_005_occlusal_upper.png': (
        'Dental occlusal view of upper arch, patient mouth open wide, '
        'large occlusal mirror reflecting maxillary arch from above, '
        'clinical ring flash lighting, all upper teeth visible including molars, '
        'black background, sharp dental macro photography, AACD documentation standard, '
        'no condensation on mirror, clean and dry dentition',
        'cartoon, blurry mirror, condensation, saliva, dark exposure, artistic effect, text, watermark',
        1216, 832, 3003),

    'odontoiatrica/dental_006_treatment_in_progress.png': (
        'Dental treatment in progress photo, dentist working on patient with dental handpiece, '
        'dental assistant retracting, modern dental operatory with overhead dental light, '
        'professional clinical setting, equipment visible including saliva ejector and mirror, '
        'realistic medical documentation photography, clinical and professional atmosphere',
        'cartoon, anime, dramatic scary perspective, blurry, blood visible, dark room, amateur photo, text, watermark',
        1216, 832, 3004),

    'odontoiatrica/dental_007_lateral_smile_45.png': (
        'Professional dental portrait, patient 3/4 view showing smile at 45 degrees, '
        'natural soft studio lighting, neutral grey background, '
        'natural genuine smile showing upper and lower teeth, cosmetic dentistry showcase, '
        'professional photography quality, sharp detail on teeth and lips, patient confident',
        'cartoon, stiff posed smile, over-retouched, blurry, dramatic shadows, colored background, text, watermark',
        1024, 1024, 3001),

    # ── VEICOLI / OFFICINA ──────────────────────────────────────────────────
    'veicoli/vehicle_001_damage_front_45.png': (
        'Professional vehicle damage documentation photo, silver sedan front-right 45 degree angle, '
        'visible front bumper damage and crumpled hood after collision, '
        'auto body shop intake photography, bright overcast daylight or diffused workshop lighting, '
        'clean concrete floor, technical documentation style, sharp focus, realistic automotive photography',
        'cartoon, dramatic shadows, studio glamour, blurry, unrealistic damage, burning fire effects, dark background, text, watermark',
        1216, 832, 4001),

    'veicoli/vehicle_002_damage_closeup.png': (
        'Close-up vehicle damage documentation, perpendicular straight-on view of rear quarter panel dent and scratch, '
        'measuring tape placed beside damage for scale reference, '
        'auto body shop documentation, diffused workshop lighting showing dent depth, '
        'technical collision repair photography, high detail, no distortion',
        'cartoon, artistic bokeh, dramatic lighting, blurry damage area, angled distortion, reflective glare, text, watermark',
        1216, 832, 4002),

    'veicoli/vehicle_003_engine_underhood.png': (
        'Vehicle underhood inspection documentation photo, car hood open showing engine bay, '
        'technician visible on right side of frame examining engine with flashlight, '
        'modern auto workshop with overhead lighting, professional workshop environment, '
        'technical inspection photography, sharp detail across engine bay, realistic automotive documentation',
        'cartoon, dark unclear photo, greasy dramatic style, blurry, artistic lighting, text, watermark',
        1216, 832, 4003),

    'veicoli/vehicle_004_repair_progress.png': (
        'Auto body repair progress documentation, technician in grey coveralls '
        'using pneumatic sanding tool on car body panel, auto body shop environment, '
        'red vehicle with visible primer grey patches where repair in progress, '
        'bright workshop fluorescent lighting, professional realistic workshop photography',
        'cartoon, anime, sparks flying dramatically, dangerous unsafe posture, blurry, dark, text, watermark',
        1216, 832, 4004),

    'veicoli/vehicle_005_before_after_repair.png': (
        'Vehicle collision repair before and after comparison, same silver SUV rear quarter panel, '
        'left panel BEFORE showing deep dent and paint scratch, '
        'right panel AFTER showing perfect repair with matching paint, '
        'identical 45 degree angle both panels, professional workshop exterior lighting, '
        'automotive repair quality documentation, realistic photography',
        'cartoon, different lighting between panels, unrealistic reflection, different angle panels, artistic filter, watermark',
        1216, 832, 4001),

    'veicoli/vehicle_006_post_repair_delivery.png': (
        'Newly repaired car ready for customer delivery, clean white sedan, '
        'professional auto body shop exterior, front-left 45 degree angle, '
        'vehicle freshly washed and polished, perfect panel alignment visible, '
        'bright daylight, professional automotive photography, quality control documentation style',
        'cartoon, dirty background, cluttered shop, dramatic shadows, blurry, artistic style, text, watermark',
        1216, 832, 4005),

    'veicoli/vehicle_007_vin_documentation.png': (
        'Close-up documentation photo of vehicle VIN plate on driver door jamb, '
        'sharp macro focus on VIN number plate, clean detail visible, '
        'car door open showing door jamb, neutral natural light, '
        'technical vehicle documentation photography, flat clear documentation style',
        'cartoon, blurry VIN, dark exposure, artistic bokeh, angled distortion, glare obscuring numbers, text overlay, watermark',
        1024, 1024, 4006),

    'veicoli/vehicle_008_dashboard_inspection.png': (
        'Vehicle interior dashboard documentation photo, close-up of car instrument cluster, '
        'driver seat perspective, showing mileage odometer reading clearly, '
        'dashboard warning lights visible, realistic automotive interior photography, '
        'bright interior lighting, technical intake documentation style, no glare on screen',
        'cartoon, blurry display, dark exposure, dramatic shadows, artistic framing, glare obscuring odometer, text, watermark',
        1216, 832, 4007),
}

print(f'Prompts caricati: {len(PROMPTS)}')
for k in PROMPTS:
    _, _, w, h, seed, *_ = PROMPTS[k]
    print(f'  {k} — {w}x{h} seed={seed}')

In [ ]:
# CELLA 4 — PRE-ENCODING T5+CLIP + CARICA PIPELINE
# Architettura v19 — ZERO bitsandbytes — validata su P100 CC6.0
import gc, torch, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from diffusers import FluxPipeline
from transformers import T5EncoderModel, CLIPTextModel, CLIPTokenizer, AutoTokenizer

# Crea directory output
from pathlib import Path
OUT = Path('/kaggle/working/f06-media')
for verticale in ['fitness', 'fisioterapia', 'odontoiatrica', 'veicoli']:
    (OUT / verticale).mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUT}')

# STEP 1: Pre-encode tutti i prompt su CPU con T5+CLIP
print('Caricamento T5-XXL su CPU (float16)...')
tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_PATH, subfolder='tokenizer_2', local_files_only=True)
t5 = T5EncoderModel.from_pretrained(
    MODEL_PATH, subfolder='text_encoder_2',
    torch_dtype=torch.float16, low_cpu_mem_usage=True, local_files_only=True)

print('Caricamento CLIP su CPU (float16)...')
tokenizer = CLIPTokenizer.from_pretrained(MODEL_PATH, subfolder='tokenizer', local_files_only=True)
clip = CLIPTextModel.from_pretrained(
    MODEL_PATH, subfolder='text_encoder',
    torch_dtype=torch.float16, low_cpu_mem_usage=True, local_files_only=True)

print(f'Encoding {len(PROMPTS)} prompt su CPU...')
all_embeds = {}
for name, (prompt, neg_prompt, w, h, seed) in PROMPTS.items():
    t5_tok = tokenizer_2(prompt, return_tensors='pt', padding='max_length',
                          max_length=256, truncation=True, add_special_tokens=True)
    with torch.no_grad():
        pe = t5(input_ids=t5_tok.input_ids).last_hidden_state.to(torch.float16).cpu()
    clip_tok = tokenizer(prompt, return_tensors='pt', padding='max_length',
                          max_length=77, truncation=True)
    with torch.no_grad():
        ppe = clip(input_ids=clip_tok.input_ids).pooler_output.to(torch.float16).cpu()
    all_embeds[name] = (pe, ppe)
    print(f'  OK: {name}')

del t5, clip, tokenizer, tokenizer_2
gc.collect()
free_vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'T5+CLIP liberati | VRAM libera: {free_vram:.2f}GB')

# STEP 2: Carica pipeline SENZA text encoders
print('Caricamento FluxPipeline float16 (no text encoders)...')
pipe = FluxPipeline.from_pretrained(
    MODEL_PATH,
    text_encoder=None,
    text_encoder_2=None,
    tokenizer=None,
    tokenizer_2=None,
    torch_dtype=torch.float16,
    local_files_only=True,
)
pipe.enable_sequential_cpu_offload()
print('Pipeline OK — sequential CPU offload attivo')

In [ ]:
# CELLA 5 — GENERA 29 IMMAGINI F06
# STEPS=20 per qualità fotografica (schnell supporta 1-50, ottimale 4 veloce / 20 qualità)
import torch, os
from pathlib import Path
from PIL import Image

STEPS = 20  # 20 per qualità realistica — usa 4 per test veloce
results = []
errors = []

for filename, (prompt, neg_prompt, w, h, seed) in PROMPTS.items():
    out_path = OUT / filename
    if out_path.exists():
        print(f'  SKIP (già esiste): {filename}')
        results.append(filename)
        continue
    print(f'Generazione [{w}x{h} seed={seed}]: {filename}...')
    try:
        pe, ppe = all_embeds[filename]
        g = torch.Generator(device='cpu').manual_seed(seed)
        result = pipe(
            prompt=None,
            prompt_embeds=pe,
            pooled_prompt_embeds=ppe,
            width=w,
            height=h,
            num_inference_steps=STEPS,
            guidance_scale=0.0,  # schnell: guidance_scale=0.0
            generator=g,
            max_sequence_length=256,
        )
        img = result.images[0]
        img.save(str(out_path))
        size_kb = os.path.getsize(str(out_path)) // 1024
        print(f'  OK — {img.size[0]}x{img.size[1]} | {size_kb}KB')
        results.append(filename)
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  ERRORE: {e}')
        errors.append((filename, str(e)))

print(f'\n=== COMPLETATO: {len(results)}/{len(PROMPTS)} immagini ===')
if errors:
    print('ERRORI:')
    for f, e in errors:
        print(f'  {f}: {e}')

In [ ]:
# CELLA 6 — PREVIEW per verticale
from IPython.display import display, Image as IPImage
from pathlib import Path

OUT = Path('/kaggle/working/f06-media')
for verticale in ['fitness', 'fisioterapia', 'odontoiatrica', 'veicoli']:
    print(f'\n{'='*50}')
    print(f'VERTICALE: {verticale.upper()}')
    print('='*50)
    folder = OUT / verticale
    if folder.exists():
        for img_path in sorted(folder.glob('*.png')):
            print(f'  {img_path.name}')
            display(IPImage(filename=str(img_path), width=700))
    else:
        print('  (nessuna immagine generata)')